---
title: 3.2.c Riesgo — Sequía y seguridad hídrica
short_title: c. Riesgo
---

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mromerot/sincelejo/blob/main/book/03_analisis/32_sequia/c_riesgo.ipynb)

**Reproducible sin red en el checkout.** La fuente preservada y sus limitaciones se describen en el capítulo 2.3.


In [1]:
from pathlib import Path
import sys, subprocess, json
candidatos = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidatos if (p/'src/sincelejo/analisis.py').exists()), None)
if ROOT is None:
    ROOT = Path('/content/sincelejo')
    if not ROOT.exists():
        subprocess.run(['git','clone','https://github.com/mromerot/sincelejo.git',str(ROOT)],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(ROOT)+'[libro]'],check=True)
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sincelejo.analisis import cargar, indicadores, deficit_trimestral, sensibilidad_calor
clima = cargar(ROOT/'data/raw/power_1991_2025.json')
anual = indicadores(clima)
evidencia = pd.read_json(ROOT/'data/evidencia.json')
plt.rcParams.update({'figure.figsize':(9,4), 'axes.spines.top':False, 'axes.spines.right':False})


## Resultado de riesgo que puede sostenerse

Se publican rachas y anomalías meteorológicas. La seguridad hídrica requiere balances y series operativas adicionales: no se infiere un agotamiento del acuífero ni un porcentaje de hogares desabastecidos.

In [2]:
local = evidencia.loc[evidencia.amenaza.eq('sequia'),['id','fecha','lugar','hallazgo','limite']]
if local.empty:
    print('No hay un hecho local incorporado en el registro para esta amenaza. No equivale a ausencia de riesgo.')
else:
    display(local)
censo=json.loads((ROOT/'data/censo2018.json').read_text())
print('Contexto demográfico: ',censo['total'],'personas censadas en 2018; no son personas expuestas a esta amenaza.')

,id,fecha,lugar,hallazgo,limite
1,E02,2024,Sistema de acueducto de Sincelejo,El PDM describe abastecimiento desde Morroa me...,"Inventario del plan, no verificación operativa..."
2,E03,2024,"Acuífero de Morroa, jurisdicción de CARSUCRE",Presión por explotación intensiva y dificultad...,No permite atribuir un descenso piezométrico a...
6,E07,2024-01-30,Sincelejo,Acciones de preparación para El Niño y abastec...,No cuantifica pérdidas ni verifica ejecución p...


Contexto demográfico:  274622 personas censadas en 2018; no son personas expuestas a esta amenaza.


## Exposición

Hogares, instituciones sanitarias, escuelas, actividades económicas y producción rural conectados a cada fuente. El inventario útil es por sistema y sector de servicio, no por proximidad al pozo.

## Vulnerabilidad y capacidades

Continuidad horaria, capacidad de almacenamiento seguro, ingresos, movilidad, calidad del agua y redundancia operativa. La vulnerabilidad agrícola incluye calendario, suelo, cultivo y acceso al agua; no se representa mediante daño de edificaciones.

In [3]:
display(pd.DataFrame([{'métrica_objetivo':'Horas sin servicio, volumen no suministrado y hogares por sector; pérdidas productivas por cultivo','datos_faltantes':'Niveles estáticos y dinámicos, extracción, recarga, demanda, pérdidas de red y continuidad','estado':'No estimada en esta edición'}]))

,métrica_objetivo,datos_faltantes,estado
0,"Horas sin servicio, volumen no suministrado y ...","Niveles estáticos y dinámicos, extracción, rec...",No estimada en esta edición


## Ruta para cuantificar

Definir unidad y periodo; obtener la intensidad en cada ubicación; contrastar el inventario; validar la relación intensidad–impacto; y propagar incertidumbre. La cifra demográfica agregada no sustituye la intersección espacial. Las pérdidas directas y los efectos por interrupción se reportarán por separado para evitar doble conteo.

**Decisión actual:** Integrar piezometría, bombeo, energía y continuidad en una misma serie.